# Abadie, Diamond & Hainmueller (2010) — California Prop 99

**Paper:** Abadie, A., Diamond, A. & Hainmueller, J. (2010). *Synthetic Control Methods for Comparative Case Studies.* JASA 105(490), 493–505. (bib key `abadie2010synthetic`)

**Design:** Synthetic control. **Data:** real ADH panel (`california_prop99.csv`, 39 states × 31 years, 1970–2000; byte-identical to `tidysynth`'s smoking dataset).

**What we reproduce:** the post-1989 gap between California and its synthetic control — ADH Figure 2 shows ≈ −19 packs/capita.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')  # headless-safe (notebooks run under nbclient in CI)
import matplotlib.pyplot as plt
import numpy as np
import statspai as sp
print('statspai', sp.__version__)

In [ ]:
df, _ = sp.replicate('abadie_2010')
print(df.shape)
df.head()

In [ ]:
# Outcome-only classical synthetic control (closest reproducible
# recipe to ADH 2010 Figure 2).
sc = sp.synth(
    data=df, outcome='cigsale', unit='state', time='year',
    treated_unit='California', treatment_time=1989,
    method='classic', placebo=False)
att = float(sc.estimate)
print(f'Average post-1989 ATT: {att:.2f} packs/capita')

In [ ]:
import pandas as pd
tab = pd.DataFrame([
    ['Avg post-1989 ATT (packs/capita)', att, -19.0,
     'ADH (2010) Figure 2 (qualitative ~ -19)'],
], columns=['quantity', 'StatsPAI', 'Paper', 'source'])
tab

In [ ]:
# Figure: observed California vs the rest-of-donor average
ca = df[df['state'] == 'California'].sort_values('year')
others = (df[df['state'] != 'California']
          .groupby('year')['cigsale'].mean())
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(ca['year'], ca['cigsale'], lw=2, label='California')
ax.plot(others.index, others.values, lw=2, ls='--',
        label='Donor average')
ax.axvline(1989, color='k', ls=':', label='Prop 99 (1989)')
ax.set_xlabel('Year'); ax.set_ylabel('Cigarette sales (packs/capita)')
ax.set_title('California Prop 99'); ax.legend()
fig.tight_layout(); fig

In [ ]:
# --- DRIFT GUARD ---
# SCM is sensitive to the predictor recipe; we pin the outcome-only
# recovery to within 0.5 of the StatsPAI reference (-19.76).
assert abs(att - (-19.7605)) < 0.5, att
# Scientific check: a sizeable negative (smoking fell) gap.
assert att < -10
print(f'OK: ADH (2010) reproduced (ATT={att:.2f}, paper ~ -19).')

**Result.** The outcome-only synthetic control recovers an average post-1989 ATT of ≈ −19.8 packs/capita, matching ADH (2010) Figure 2's ≈ −19. Modern refinements (synthdid, augmented SCM) that reduce predictor-recipe sensitivity are in the modern track of `sp.replicate('abadie_2010')`.